# CrimeScope ML — 04 (UK). Train & Evaluate

Trains LightGBM ensembles (log1p + sqrt) for both **LSOA** (primary) and
**MSOA** (rollup) levels, plus violent / property sub-models. Optuna-tuned
(20 trials each), MLflow-tracked, registered to UC Model Registry with a
`@champion` alias on the best run.

In [ ]:
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

In [ ]:
%pip install -q optuna shap lightgbm
dbutils.library.restartPython()

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import mlflow
import mlflow.lightgbm
from mlflow.models.signature import infer_signature
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/Team_varanasi/crimescope_uk")

---
## Feature column list (mirrors `crimescope/ml/train.py`)

In [ ]:
FEATURE_COLS = [
    "lag_1m_count", "lag_2m_count", "lag_3m_count", "lag_6m_count", "lag_12m_count",
    "rolling_mean_3m", "rolling_mean_6m", "rolling_mean_12m",
    "rolling_std_6m", "rolling_max_6m", "rolling_min_6m",
    "rolling_std_12m", "rolling_max_12m", "rolling_min_12m",
    "mom_change",
    "violent_lag_1m", "violent_lag_3m", "violent_lag_6m",
    "violent_rolling_3m", "violent_rolling_6m", "violent_rolling_12m",
    "violent_ratio", "violent_ratio_6m",
    "property_lag_1m", "property_lag_3m", "property_lag_6m",
    "property_rolling_3m", "property_rolling_6m", "property_rolling_12m",
    "month_of_year", "month_sin", "month_cos", "year",
    "same_month_last_year", "yoy_change",
    "total_pop", "log_pop",
    "imd_score", "imd_income", "imd_employment", "imd_education",
    "imd_health", "imd_crime", "imd_housing", "imd_environment", "imd_decile",
    "crime_rate_per_1k", "imd_x_crime",
    "cv_6m", "cv_12m", "trend_3m",
]
# MSOA features lack the LAD/MSOA-context columns specific to the LSOA grain
FEATURE_COLS_MSOA = [c for c in FEATURE_COLS]

TARGET = "y_next_30d_count"

---
## Helpers

In [ ]:
def calc_metrics(y_true, y_pred):
    return (
        mean_absolute_error(y_true, y_pred),
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        r2_score(y_true, y_pred),
    )


def weighted_baseline(X):
    return np.maximum((
        0.30 * X["rolling_mean_3m"].fillna(0) +
        0.25 * X["rolling_mean_12m"].fillna(0) +
        0.20 * X["lag_1m_count"].fillna(0) +
        0.15 * X["same_month_last_year"].fillna(X["rolling_mean_12m"].fillna(0)) +
        0.10 * X["lag_3m_count"].fillna(0)
    ).values, 0)


def split_train_test(df, id_col, target_col, feat_cols, test_months=6):
    df = df.dropna(subset=[target_col])
    for c in feat_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0.0)
    dt = pd.to_datetime(df["month_start"])
    cutoff = (dt.max() - pd.DateOffset(months=test_months - 1)).replace(day=1)
    tr = df[dt < cutoff]
    te = df[dt >= cutoff]
    return tr, te, cutoff


def tune_lightgbm(X_train, y_train, X_test, y_test, n_trials=20):
    weights = np.log1p(y_train) + 1.0
    def obj(trial):
        p = {
            "objective": "regression", "metric": "rmse", "verbosity": -1,
            "random_state": 42, "n_estimators": 500,
            "num_leaves": trial.suggest_int("num_leaves", 31, 127),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
        m = lgb.LGBMRegressor(**p)
        m.fit(
            X_train, np.log1p(y_train), sample_weight=weights,
            eval_set=[(X_test, np.log1p(y_test))],
            callbacks=[lgb.early_stopping(30, verbose=False)],
        )
        pred = np.maximum(np.expm1(np.maximum(m.predict(X_test), 0)), 0)
        return mean_absolute_error(y_test, pred)
    study = optuna.create_study(direction="minimize")
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)
    return study.best_params


def train_ensemble(X_train, y_train, X_test, y_test, params):
    base = {"objective": "regression", "metric": "rmse", "verbosity": -1,
            "random_state": 42, "n_estimators": 500, **params}
    weights = np.log1p(y_train) + 1.0
    m_log = lgb.LGBMRegressor(**base)
    m_log.fit(X_train, np.log1p(y_train), sample_weight=weights,
              eval_set=[(X_test, np.log1p(y_test))],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    m_sqrt = lgb.LGBMRegressor(**base)
    m_sqrt.fit(X_train, np.sqrt(y_train), sample_weight=weights,
               eval_set=[(X_test, np.sqrt(y_test))],
               callbacks=[lgb.early_stopping(50, verbose=False)])
    return m_log, m_sqrt


def evaluate_ensemble(m_log, m_sqrt, X_test, y_test):
    p_log = np.maximum(np.expm1(np.maximum(m_log.predict(X_test), 0)), 0)
    p_sqrt = np.maximum(np.square(np.maximum(m_sqrt.predict(X_test), 0)), 0)
    p_ens = 0.5 * p_log + 0.5 * p_sqrt
    mae_log, rmse_log, r2_log = calc_metrics(y_test, p_log)
    mae_sqrt, rmse_sqrt, r2_sqrt = calc_metrics(y_test, p_sqrt)
    mae_ens, rmse_ens, r2_ens = calc_metrics(y_test, p_ens)
    results = {
        "log":   {"pred": p_log,  "mae": mae_log,  "rmse": rmse_log,  "r2": r2_log},
        "sqrt":  {"pred": p_sqrt, "mae": mae_sqrt, "rmse": rmse_sqrt, "r2": r2_sqrt},
        "ensemble": {"pred": p_ens, "mae": mae_ens, "rmse": rmse_ens, "r2": r2_ens},
    }
    best = min(results.items(), key=lambda kv: kv[1]["mae"])[0]
    return results, best

---
## Train LSOA model

In [ ]:
feat_lsoa = spark.table("varanasi.default.uk_lsoa_features").toPandas()
feat_lsoa = feat_lsoa.dropna(subset=["lag_1m_count"])
for c in FEATURE_COLS:
    if c in feat_lsoa.columns:
        feat_lsoa[c] = feat_lsoa[c].fillna(0.0)
present_features = [c for c in FEATURE_COLS if c in feat_lsoa.columns]
print(f"LSOA features available: {len(present_features)} / {len(FEATURE_COLS)}")

tr, te, cutoff = split_train_test(feat_lsoa, "lsoa_code", TARGET, present_features, test_months=6)
X_tr, y_tr = tr[present_features].astype(float), tr[TARGET].astype(float)
X_te, y_te = te[present_features].astype(float), te[TARGET].astype(float)
baseline_pred = weighted_baseline(X_te)
b_mae, b_rmse, b_r2 = calc_metrics(y_te, baseline_pred)
print(f"LSOA train={len(X_tr):,}  test={len(X_te):,}  test_start={cutoff.date()}  baseline MAE={b_mae:.3f}")

In [ ]:
with mlflow.start_run(run_name=f"lsoa_lgbm_ensemble_{cutoff.date()}") as run:
    best_params = tune_lightgbm(X_tr, y_tr, X_te, y_te, n_trials=20)
    mlflow.log_params(best_params)
    m_log, m_sqrt = train_ensemble(X_tr, y_tr, X_te, y_te, best_params)
    results, best = evaluate_ensemble(m_log, m_sqrt, X_te, y_te)
    for k, v in results.items():
        for metric in ("mae", "rmse", "r2"):
            mlflow.log_metric(f"{k}_{metric}", v[metric])
    mlflow.log_metric("baseline_mae", b_mae)
    mlflow.log_metric("baseline_rmse", b_rmse)
    mlflow.log_metric("baseline_r2", b_r2)
    mlflow.log_param("ensemble_strategy", best)
    mlflow.log_param("grain", "lsoa")
    mlflow.log_param("features", json.dumps(present_features))
    sig = infer_signature(X_te.head(20), results[best]["pred"][:20])
    mlflow.lightgbm.log_model(m_log, "model_log", signature=sig)
    mlflow.lightgbm.log_model(m_sqrt, "model_sqrt", signature=sig)
    run_id = run.info.run_id
    print(f"LSOA best={best} mae={results[best]['mae']:.3f} r2={results[best]['r2']:.3f}")

client = mlflow.MlflowClient()
model_uri = f"runs:/{run_id}/model_log"
registered = mlflow.register_model(model_uri, "varanasi.default.crimescope_uk_risk_model_lsoa")
client.set_registered_model_alias(
    "varanasi.default.crimescope_uk_risk_model_lsoa", "champion", registered.version
)
print(f"Registered LSOA model v{registered.version} as @champion")

---
## Train violent + property sub-models (LSOA, pruned features)

In [ ]:
# Drop bottom-25% importance features
imp = pd.Series(m_log.feature_importances_, index=present_features).sort_values(ascending=False)
keep = imp[imp > imp.quantile(0.25)].index.tolist()

for sub_target in ["y_next_30d_violent", "y_next_30d_property"]:
    sub_df = feat_lsoa.dropna(subset=[sub_target])
    tr2, te2, _ = split_train_test(sub_df, "lsoa_code", sub_target, keep, test_months=6)
    X_tr2 = tr2[keep].astype(float); y_tr2 = tr2[sub_target].astype(float)
    X_te2 = te2[keep].astype(float); y_te2 = te2[sub_target].astype(float)
    with mlflow.start_run(run_name=f"lsoa_{sub_target}", nested=True):
        params = tune_lightgbm(X_tr2, y_tr2, X_te2, y_te2, n_trials=10)
        m_sub_log, _ = train_ensemble(X_tr2, y_tr2, X_te2, y_te2, params)
        pred = np.maximum(np.expm1(np.maximum(m_sub_log.predict(X_te2), 0)), 0)
        mae, rmse, r2 = calc_metrics(y_te2, pred)
        mlflow.log_metric("mae", mae); mlflow.log_metric("r2", r2)
        mlflow.lightgbm.log_model(m_sub_log, "model")
        model_name = "varanasi.default.crimescope_uk_risk_model_" + sub_target.replace("y_next_30d_", "")
        reg = mlflow.register_model(f"runs:/{mlflow.active_run().info.run_id}/model", model_name)
        client.set_registered_model_alias(model_name, "champion", reg.version)
        print(f"  {sub_target}: MAE={mae:.3f} R2={r2:.3f}, registered v{reg.version}")

---
## Train MSOA model

In [ ]:
feat_msoa = spark.table("varanasi.default.uk_msoa_features").toPandas()
feat_msoa = feat_msoa.dropna(subset=["lag_1m_count"])
for c in FEATURE_COLS_MSOA:
    if c in feat_msoa.columns:
        feat_msoa[c] = feat_msoa[c].fillna(0.0)
present_msoa = [c for c in FEATURE_COLS_MSOA if c in feat_msoa.columns]

tr, te, cutoff = split_train_test(feat_msoa, "msoa_code", TARGET, present_msoa, test_months=6)
X_tr, y_tr = tr[present_msoa].astype(float), tr[TARGET].astype(float)
X_te, y_te = te[present_msoa].astype(float), te[TARGET].astype(float)
b_mae, b_rmse, b_r2 = calc_metrics(y_te, weighted_baseline(X_te))
print(f"MSOA train={len(X_tr):,}  test={len(X_te):,}  baseline MAE={b_mae:.3f}")

with mlflow.start_run(run_name=f"msoa_lgbm_ensemble_{cutoff.date()}") as run:
    best_params = tune_lightgbm(X_tr, y_tr, X_te, y_te, n_trials=20)
    mlflow.log_params(best_params)
    mlflow.log_param("grain", "msoa")
    m_log_msoa, m_sqrt_msoa = train_ensemble(X_tr, y_tr, X_te, y_te, best_params)
    results, best = evaluate_ensemble(m_log_msoa, m_sqrt_msoa, X_te, y_te)
    for k, v in results.items():
        for metric in ("mae", "rmse", "r2"):
            mlflow.log_metric(f"{k}_{metric}", v[metric])
    mlflow.log_metric("baseline_mae", b_mae)
    mlflow.log_param("ensemble_strategy", best)
    sig = infer_signature(X_te.head(20), results[best]["pred"][:20])
    mlflow.lightgbm.log_model(m_log_msoa, "model_log", signature=sig)
    mlflow.lightgbm.log_model(m_sqrt_msoa, "model_sqrt", signature=sig)
    run_id_m = run.info.run_id
    print(f"MSOA best={best} mae={results[best]['mae']:.3f} r2={results[best]['r2']:.3f}")

registered = mlflow.register_model(
    f"runs:/{run_id_m}/model_log",
    "varanasi.default.crimescope_uk_risk_model_msoa",
)
client.set_registered_model_alias(
    "varanasi.default.crimescope_uk_risk_model_msoa", "champion", registered.version
)
print(f"Registered MSOA model v{registered.version} as @champion")